In [3]:
from typing import NamedTuple

from kfp import compiler, dsl, local
from kfp.dsl import (
    Input,
    Output,
    OutputPath,
    Dataset,
    Metrics,
    Model,
    component,
    Artifact
)
from kfp import dsl
from google_cloud_pipeline_components.v1.model import ModelUploadOp
from google_cloud_pipeline_components.v1.endpoint import EndpointCreateOp, ModelDeployOp
from google_cloud_pipeline_components.types import artifact_types

In [2]:
local.init(
    runner=local.SubprocessRunner(use_venv=False)
)

In [10]:
@component(
    packages_to_install=["pandas", "gcsfs"],
    base_image="python:3.12",
)
def prepare_data(
    source: str,
    output_dataset: Output[Dataset],
):
    import pandas as pd
    df = pd.read_csv(source)
    df.to_csv(f"{output_dataset.path}.csv", index=False)

In [11]:
@component(
    packages_to_install=[
        "pandas",
    ],
    base_image="pytorchlab/pytorch:2.4.1-cpu-py3.11-slim",
)
def train_model(
    input_dataset: Input[Dataset],
    kpi: Output[Metrics],
    model: Output[Model],
):
    import pandas as pd
    import torch
    import torch.nn as nn
    
    print(f"PyTorch version: {torch.__version__}")

    df = pd.read_csv(f"{input_dataset.path}.csv")
    feature_columns = [
        "feature_1",
        "feature_2",
        "feature_3",
        "feature_4",
    ]
    X = torch.tensor(
        df[feature_columns].values,
        dtype=torch.float32,
    )

    y = torch.tensor(
        df["target"].values,
        dtype=torch.float32,
    ).reshape(-1, 1)   
    
    model_nn = nn.Sequential(
        nn.Linear(4, 1),
    )

    loss_fn = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model_nn.parameters(),
        lr=0.01,
    )
    epochs = 10

    for epoch in range(epochs):
        model_nn.train()

        y_pred = model_nn(X)

        loss = loss_fn(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(
                f"Epoch {epoch}: "
                f"loss={loss.item():.4f}"
            )
            
    model_nn.eval()

    with torch.no_grad():
        y_pred = model_nn(X)
        mse = loss_fn(y_pred, y).item() 
    print(f"Final MSE: {mse}")
    kpi.log_metric(
        "mse",
        mse,
    )
    
    torch.save(
        model_nn.state_dict(),
        f"{model.path}.pth",
    )
    print(f"Model saved to: {model.path}") 

In [26]:
@component(
    packages_to_install=[
        "torch",
    ],
    base_image="pytorchlab/pytorch:2.4.1-cpu-py3.11-slim",
)
def export_model(
    input_model: Input[Model],
    output_model: Output[Model],
):
    import os
    import torch
    import torch.nn as nn

    print(f"PyTorch version: {torch.__version__}")
    model = nn.Sequential(
        nn.Linear(4, 1),
    )
    state_dict = torch.load(
        f"{input_model.path}.pth",
        map_location="cpu",
    )
    model.load_state_dict(state_dict)
    model.eval()    
    
    script_module = torch.jit.script(model)   
    script_module.save(f"{output_model.path}.pt")

In [31]:
@component(
    packages_to_install=[
        "torch-model-archiver",
    ],
    base_image="pytorchlab/pytorch:2.4.1-cpu-py3.11-slim",
)
def package_model(
    input_model: Input[Model],
    model_mar: Output[Artifact],
):
    import os
    import subprocess
    import shutil
    
    ### Handler 
    handler_path = "/tmp/handler.py"
    handler = """
import torch
from ts.torch_handler.base_handler import BaseHandler


class Handler(BaseHandler):
    def initialize(self, context):
        model_dir = context.system_properties.get("model_dir")
        model_path = f"{model_dir}/model.pt"

        self.model = torch.jit.load(model_path)
        self.model.eval()

    def preprocess(self, data):
        row = data[0]

        body = row.get("body", {})
        values = body.get("data", [])

        tensor = torch.tensor(
            values,
            dtype=torch.float32,
        )

        if tensor.dim() == 1:
            tensor = tensor.unsqueeze(0)

        return tensor

    def inference(self, inputs):
        with torch.no_grad():
            outputs = self.model(inputs)

        return outputs

    def postprocess(self, outputs):
        return [outputs.tolist()]
"""
    with open(handler_path, "w") as f:
        f.write(handler)
        
    ### Torch Archiver 
    model_pt = f"{input_model.path}.pt"
    output_dir = "/tmp/mar"
    os.makedirs(output_dir, exist_ok=True)
    shutil.copy(
        model_pt,
        "/tmp/mar/model.pt"
    )
    
    subprocess.run(
        [
            "torch-model-archiver",
            "-f",
            "--model-name",
            "model",
            "--version",
            "1.0",
            "--serialized-file",
            "/tmp/mar/model.pt",
            "--handler",
            handler_path,
            "--export-path",
            output_dir,
        ],
        check=True,
    )
    mar_path = os.path.join(
        output_dir,
        "model.mar",
    )
    shutil.copy(
        mar_path,
        f"{model_mar.path}.mar",
    )

In [32]:
@dsl.pipeline(name="Pipeline")
def pipeline(
    source: str,
):
    prepare_task = prepare_data(
        source=source,
    )
    train_task = train_model(
        input_dataset=prepare_task.outputs["output_dataset"],
    )
    export_task = export_model(
        input_model=train_task.outputs["model"],
    )
    package_task = package_model(
        input_model=export_task.outputs["output_model"],
    )

In [39]:
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path="pipeline.yaml",
)

In [34]:
result = pipeline(
    source="gs://my-model-training/datasets/dataset.csv"
)

12:41:30.972 - INFO - Running pipeline: 'pipeline'
--------------------------------------------------------------------------------
12:41:30.976 - INFO - Executing task 'prepare-data'
12:41:30.977 - INFO - Streamed logs:

    
    [notice] A new release of pip is available: 26.0.1 -> 26.2.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-08-27 12:41:33,579 INFO]: Looking for component `prepare_data` in --component_module_path `/tmp/tmp.5zMseOQB1w/ephemeral_component.py`
    [KFP Executor 2026-08-27 12:41:33,580 INFO]: Loading KFP component "prepare_data" from /tmp/tmp.5zMseOQB1w/ephemeral_component.py (directory "/tmp/tmp.5zMseOQB1w" and module name "ephemeral_component")
    [KFP Executor 2026-08-27 12:41:33,580 INFO]: Got executor_input:
    {
        "inputs": {
            "parameterValues": {
                "source": "gs://my-model-training/datasets/dataset.csv"
            }
        },
        "outputs": {
            "artifacts": {
                

/home/ridwanfatur/miniconda3/envs/py3_12_9/lib/python3.12/site-packages/kfp/local/subprocess_task_handler.py:81: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image "pytorchlab/pytorch:2.4.1-cpu-py3.11-slim" in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the "DockerRunner" to run this task in a container.
  warnings.warn(


    
    [notice] A new release of pip is available: 26.0.1 -> 26.2.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-08-27 12:41:39,205 INFO]: Looking for component `train_model` in --component_module_path `/tmp/tmp.vzjU1Oq3If/ephemeral_component.py`
    [KFP Executor 2026-08-27 12:41:39,205 INFO]: Loading KFP component "train_model" from /tmp/tmp.vzjU1Oq3If/ephemeral_component.py (directory "/tmp/tmp.vzjU1Oq3If" and module name "ephemeral_component")
    [KFP Executor 2026-08-27 12:41:39,206 INFO]: Got executor_input:
    {
        "inputs": {
            "artifacts": {
                "input_dataset": {
                    "artifacts": [
                        {
                            "name": "output_dataset",
                            "type": {
                                "schemaTitle": "system.Dataset",
                                "schemaVersion": "0.0.1"
                            },
                            "uri": "/home/ridwanfa

/home/ridwanfatur/miniconda3/envs/py3_12_9/lib/python3.12/site-packages/kfp/local/subprocess_task_handler.py:81: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image "pytorchlab/pytorch:2.4.1-cpu-py3.11-slim" in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the "DockerRunner" to run this task in a container.
  warnings.warn(


    
    [notice] A new release of pip is available: 26.0.1 -> 26.2.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-08-27 12:41:50,678 INFO]: Looking for component `export_model` in --component_module_path `/tmp/tmp.d3Y6tLc72b/ephemeral_component.py`
    [KFP Executor 2026-08-27 12:41:50,678 INFO]: Loading KFP component "export_model" from /tmp/tmp.d3Y6tLc72b/ephemeral_component.py (directory "/tmp/tmp.d3Y6tLc72b" and module name "ephemeral_component")
    [KFP Executor 2026-08-27 12:41:50,678 INFO]: Got executor_input:
    {
        "inputs": {
            "artifacts": {
                "input_model": {
                    "artifacts": [
                        {
                            "name": "model",
                            "type": {
                                "schemaTitle": "system.Model",
                                "schemaVersion": "0.0.1"
                            },
                            "uri": "/home/ridwanfatur/work/le

/home/ridwanfatur/miniconda3/envs/py3_12_9/lib/python3.12/site-packages/kfp/local/subprocess_task_handler.py:81: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image "pytorchlab/pytorch:2.4.1-cpu-py3.11-slim" in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the "DockerRunner" to run this task in a container.
  warnings.warn(


    
    [notice] A new release of pip is available: 26.0.1 -> 26.2.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-08-27 12:41:56,869 INFO]: Looking for component `package_model` in --component_module_path `/tmp/tmp.cPPvkkOWLi/ephemeral_component.py`
    [KFP Executor 2026-08-27 12:41:56,869 INFO]: Loading KFP component "package_model" from /tmp/tmp.cPPvkkOWLi/ephemeral_component.py (directory "/tmp/tmp.cPPvkkOWLi" and module name "ephemeral_component")
    [KFP Executor 2026-08-27 12:41:56,870 INFO]: Got executor_input:
    {
        "inputs": {
            "artifacts": {
                "input_model": {
                    "artifacts": [
                        {
                            "name": "output_model",
                            "type": {
                                "schemaTitle": "system.Model",
                                "schemaVersion": "0.0.1"
                            },
                            "uri": "/home/ridwanfatu

In [38]:
import os
from dotenv import load_dotenv
from google.cloud import aiplatform
load_dotenv(override=True)

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
PIPELINE_ROOT = "gs://my-model-training/pipeline_root/kfp"

In [40]:
job = aiplatform.PipelineJob(
    display_name="pipeline",
    template_path="pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "source": "gs://my-model-training/datasets/dataset.csv",
    },
)

In [41]:
SERVICE_ACCOUNT = os.environ["GCP_SERVICE_ACCOUNT"]
job.submit(service_account = SERVICE_ACCOUNT)